In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import gymnasium as gym
from google import genai

from gym_env import SalesNegotiationEnv
import dotenv
import os

from pydantic import BaseModel, Field
from typing import Literal
from google.genai import types

dotenv.load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

In [3]:
agent_prompt = """ You are an elite Sales Representative selling premium boots to a retail buyer. \n
Objective: Maximize your total profit. You must balance the cost of your time, the cost of incentives, and the risk of the customer walking away. \n

Financial Stakes: 
**Labor Cost**: Every round of negotiation costs you 2% of your profit. \n
**Walk-away Risk: After the first few exchanges, the customer may lose patience and leave at any time. If they leave, you get zero profit and a penalty for customer's unsatisfaction, which will be 20% of your profit. \n

- There are four types of objections:
    - A: Quality Issue. Incentive is to provide extended warranty. The cost of this incentive is 20% of the profit.
    - B: Logistics Issue. Incentive is to provide free shipping. The cost of this incentive is 50% of the profit.
    - C: Price Issue. Incentive is to provide a discount. The cost of this incentive is 70% of the profit.
    - D: General. No incentive can be provided for this objection.
Rule: Objection A, B, and C can only be raised once per negotiation. Objection D can be raised multiple times.

At each round, the customer will raise an objection. You will be given the type of objection. You must respond to the objection with one of the following:

- action 0: [PERSUADE]: No additional cost. Small chance of instant sale. 
    - If not sold, there is equal chance that the objection is resolved or not. 
    - If not resolved and you have not used an incentive, you can do so in the next round.
- action 1: [INCENTIVE]: High chance of instant sale at a cost. Higher cost = higher success probability.
    - LIMIT: You can only use an incentive once for the same objection. 
    - RESTRICTION: Forbidden for General (Category D) objections.
- action 2: [EXIT]: End the negotiation immediately to cut your losses and stop labor costs.

Expected Output Format:
'action': 0, 1 or 2.
'text': Response to the objection. Use a professional tone.
'thought': Internal reasoning for choosing Persuade/Incentive/Exit.

"""

class AgentResponse(BaseModel):
    action: Literal['0', '1', '2']
    text: str
    thought: str = Field(description="Internal reasoning for choosing Persuade/Incentive/Exit.")

In [4]:
customer_prompt = """
# ROLE
You are a customer shopping for boots. You are in a negotiation with an agent.
You will be given the type of the response and your task is only to generate the message to the agent.

# OBJECTION THEMES
- A (Quality): You are worried about the leather durability or craftsmanship.
- B (Logistics): You are worried about shipping related issues.
- C (Price): You find the cost high for your current quarterly budget.
- D (General/Misc): You have various smaller concerns (style, packaging, or general hesitation).

# RESPONSE GUIDELINES
1. If the result is "CONCERN RESOLVED": Acknowledge the agent's point briefly, but then pivot immediately to the NEW objection provided.
2. If the result is "UNCONVINCED": Stick to your guns. Reiterate the current objection with more frustration or skepticism.
3. If the result is "SALE CLOSED": Express satisfaction and agree to sign the deal.
4. If the result is "EXIT": End the conversation dismissively.
"""

In [5]:

env = SalesNegotiationEnv()

def simulation_gemini(client, agent_model = "gemini-2.5-flash", verbose = False):
    agent_chat = client.chats.create(
        model=agent_model,
        config=types.GenerateContentConfig(
            system_instruction=agent_prompt,
            response_mime_type="application/json",
            response_schema=AgentResponse
        )
    )
    customer_chat = client.chats.create(
        model="gemini-2.5-flash",
        config=types.GenerateContentConfig(
            system_instruction=customer_prompt
        )
    )

    current_round = 0
    obs, _ = env.reset()
    episode_reward = 0
    thought_history = []
    terminated = False
    truncated = False

    while (current_round < env.max_round) and (not terminated) and (not truncated):
        current_round += 1
        obs_dict = env.unpack_obs(obs)
        topic = obs_dict['topic_history'][current_round - 1]
        topic_text = env._index_to_topic(topic)
        previous_objection_resolved = bool(obs_dict['previous_objection_resolved'])

        customer_state_str = f"Current Round: {current_round}, Objection: {topic_text}, Previous Objection Resolved: {previous_objection_resolved}"
        if current_round == 1:
            init_msg = "This is the beginning of the conversation. There is no previous dialogue."
            customer_state_str = init_msg + "\n" + customer_state_str

        customer_raw = customer_chat.send_message(customer_state_str)
        customer_response = customer_raw.candidates[0].content.parts[0].text

        if verbose:
            print(f"Customer Response: {customer_response}")

        state_str = f"Current Round: {current_round}, Objection: {topic_text}, Incentive Used: {obs_dict['incentive_used']}, Previous Objection Resolved: {previous_objection_resolved}"
        agent_raw = agent_chat.send_message(state_str + f"\n\n Customer said: {customer_response}")
        agent_raw = agent_raw.parsed

        agent_action = int(agent_raw.action)
        agent_text = agent_raw.text
        agent_thought = agent_raw.thought
        thought_history.append(agent_thought)

        if verbose:
            print("Agent Message: ", agent_text, "\n\n", "Agent Thought: ", agent_thought, "\n\n")

        obs, reward, terminated, truncated, info = env.step(agent_action)
        episode_reward += reward
       
    result_dict = env.unpack_obs(obs)
    action_history = result_dict['action_history']
    
    if verbose:
        if terminated:
            print(f"Negotiation completed. Total Reward: {episode_reward}")
        elif truncated:
            print(f"Negotiation truncated. Total Reward: {episode_reward}")
        else:
            print(f"This should not happen. Total Reward: {episode_reward}")

    return episode_reward, action_history, thought_history
    

test = False
if test:
    env = SalesNegotiationEnv()
    reward, action, thoughts = simulation_gemini(client, verbose = False)
    print(f"Episode Reward: {reward}. \n\n")
    print(f"Action History: {action - 1}. \n\n")

In [19]:
from tqdm import tqdm

test_result = []
for i in tqdm(range(20), desc="Running Simulations"):
    test_id = i
    env = SalesNegotiationEnv(p_factor = 1.5)
    reward, action, thoughts = simulation_gemini(client, verbose = False)
    round = (action > 0).sum()
    test_result.append({
        'test_id': test_id, 
        'reward': reward, 
        'action': action + 1, 
        'total_round': round, 
        'thoughts': thoughts})

test_result_df = pd.DataFrame(test_result)

Running Simulations: 100%|██████████| 20/20 [08:59<00:00, 26.99s/it]


In [21]:
import time

time_stamp = time.strftime("%Y%m%d_%H%M%S")
test_result_df = pd.DataFrame(test_result)
test_result_df.to_csv(f'test_result_{time_stamp}_gemini25flash.csv', index=False)
print('average reward: ', np.array(test_result_df['reward']).mean())
print('conversion rate: ', np.array(test_result_df['reward'] > 0).mean())

average reward:  2.24
conversion rate:  0.5


In [22]:
import time

test_result = []
for i in tqdm(range(30), desc="Running Simulations"):
    test_id = i
    env = SalesNegotiationEnv()
    reward, action, thoughts = simulation_gemini(client, agent_model = "gemini-3-pro-preview", verbose = False)
    round = (action > 0).sum()
    test_result.append({
        'test_id': test_id, 
        'reward': reward, 
        'action': action + 1, 
        'total_round': round, 
        'thoughts': thoughts})

test_result_df = pd.DataFrame(test_result)

time_stamp = time.strftime("%Y%m%d_%H%M%S")
test_result_df = pd.DataFrame(test_result)
test_result_df.to_csv(f'test_result_{time_stamp}_gemini3pro.csv', index=False)
print('average reward: ', np.array(test_result_df['reward']).mean())
print('conversion rate: ', np.array(test_result_df['reward'] > 0).mean())


Running Simulations: 100%|██████████| 30/30 [47:35<00:00, 95.19s/it] 

average reward:  1.1266666666666667
conversion rate:  0.43333333333333335


In [ ]:
import os

def model_predict(model, obs):
    if obs[int(5 + obs[0])] == 4:
        if model.predict_value(np.expand_dims(obs, axis=0), 0) > model.predict_value(np.expand_dims(obs, axis=0), 2):
            return 0, model.predict_value(np.expand_dims(obs, axis=0), 0)
        else:
            return 2, model.predict_value(np.expand_dims(obs, axis=0), 2)
    elif bool(obs[4]):
        if model.predict_value(np.expand_dims(obs, axis=0), 0) > model.predict_value(np.expand_dims(obs, axis=0), 2):
            return 0, model.predict_value(np.expand_dims(obs, axis=0), 0)
        else:
            return 2, model.predict_value(np.expand_dims(obs, axis=0), 2)
    else:
        return model.predict(np.expand_dims(obs, axis=0))[0], model.predict_value(np.expand_dims(obs, axis=0), model.predict(np.expand_dims(obs, axis=0))[0])
    

def ensemble_model_predict(models, obs, top_k = 5):
    models_predictions = []
    for model in models:
        action, q_value = model_predict(model, obs)
        models_predictions.append((action, q_value))
    sorted_preds = sorted(models_predictions, key=lambda x: x[1], reverse=True)
    trimmed_preds = sorted_preds[:top_k]

    actions = [action for action, q_value in trimmed_preds]
    # Majority vote on the actions
    from collections import Counter
    action_counts = Counter(actions)
    # Pick the most common action; in case of ties, pick the smallest action number
    majority_action = min([a for a, count in action_counts.items() if count == max(action_counts.values())])
    # Optionally: pick the mean q_value of those who voted for the chosen action
    q_values_majority = [q for (a, q) in trimmed_preds if a == majority_action]
    avg_q_value = np.mean(q_values_majority) if q_values_majority else 0
    return majority_action, avg_q_value


test = False

models = []
for model_num in range(38, 53):
    model_num = model_num * 1000
    model_path = os.path.join(log_folder, f"model_{model_num}.d3")
    model = d3rlpy.load_learnable(model_path)
    models.append(model)

if test:
    obs, _ = env.reset()
    ensemble_action, ensemble_q_value = ensemble_model_predict(models, obs)
    print(f"Ensemble Action: {ensemble_action}, Ensemble Q Value: {ensemble_q_value}")


In [ ]:
def evaluate_ensemble_model(models, env, n_episodes=100):
    test_record = []
    for test_id in tqdm(range(n_episodes), desc="Evaluating Models"):
        obs, _ = env.reset()
        episode_reward = 0
        done = False
        while not done:
            action, _ = ensemble_model_predict(models, obs)
            obs, reward, terminated, truncated, _ = env.step(action)
            episode_reward += reward
            done = terminated or truncated
        
        test_record.append({"test_id": test_id, "reward": episode_reward, "round": obs[0] - 1})
    
    test_record_df = pd.DataFrame(test_record)

    test_average_reward = np.array(test_record_df["reward"]).mean()
    test_conversion_rate = np.array(np.array(test_record_df["reward"]) > 0).mean()
    test_average_round = np.array(test_record_df["round"]).mean()

    return test_average_reward, test_conversion_rate, test_average_round

reward, conversion_rate, average_round = evaluate_ensemble_model(models, env, 100)
print(f"Average reward: {reward}\n\nConversion rate: {conversion_rate}\n\nAverage round: {average_round}")


Evaluating Models: 100%|██████████| 100/100 [00:06<00:00, 15.63it/s]


Average reward: 0.4380000000000001

Conversion rate: 0.37

Average round: 5.769999980926514
